# Validation Beyond One Illustrative Case

This notebook gathers and extends the code needed to strengthen the study's validation beyond one illustrative case.

It reuses/adapts code from the uploaded notebooks:

- `01_whole_pipeline.ipynb`: strict LLM extraction, quote verification, LLM-as-Judge, Event Registry event graph construction.
- `03_Merging_CG_EventRegistry_Timing.ipynb` and `04_test_the_process.ipynb`: filtering the causal graph by event-graph features, merging causal and event graphs, and time-window analysis.
- `internal_methods.py`: reusable graph-merging and explanation utilities.

The notebook produces CSV and LaTeX tables for:

1. Event-graph coverage across saved runs.
2. Causal-event graph statistics across available prediction instances.
3. Coverage of model-important features by event evidence.
4. Saved LLM-as-Judge confirmed-link statistics.
5. Optional audit-enabled re-run to measure quote-check and judge rejection rates.

The notebook does **not** call an LLM unless `RUN_LLM_AUDIT = True` is set in Section 9.

In [1]:
from __future__ import annotations

import ast
import difflib
import json
import os
import re
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 160)

## 1. Locate project artifacts

The original notebooks use relative folders such as `../enriched_knowledge_graph`, `../causal_graph/sub_causal_graph`, `../importance_df`, `../rows`, `../knowledge_graph`, `../llm_response`, and `../articles`. This cell tries to locate the project root automatically. If needed, set `PROJECT_ROOT` manually.

In [2]:
def find_project_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    candidates = [start] + list(start.parents)
    expected_dirs = ['enriched_knowledge_graph', 'causal_graph', 'importance_df', 'rows', 'knowledge_graph', 'llm_response', 'articles', 'data']
    best, best_score = start, -1
    for c in candidates:
        score = sum((c / d).exists() for d in expected_dirs)
        if score > best_score:
            best, best_score = c, score
    return best

PROJECT_ROOT = find_project_root()
# If auto-detection fails, uncomment and set manually:
# PROJECT_ROOT = Path('/path/to/your/project/root')

PATHS = {
    'enriched_kg': PROJECT_ROOT / 'enriched_knowledge_graph',
    'causal_subgraphs': PROJECT_ROOT / 'causal_graph' / 'sub_causal_graph',
    'importance': PROJECT_ROOT / 'importance_df',
    'rows': PROJECT_ROOT / 'rows',
    'knowledge_graph_runs': PROJECT_ROOT / 'knowledge_graph',
    'llm_response': PROJECT_ROOT / 'llm_response',
    'articles': PROJECT_ROOT / 'articles',
    'validation_outputs': PROJECT_ROOT / 'validation_outputs',
}
PATHS['validation_outputs'].mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
for name, path in PATHS.items():
    print(f'{name:>20}: {path} | exists={path.exists()}')

PROJECT_ROOT: c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials
         enriched_kg: c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\enriched_knowledge_graph | exists=True
    causal_subgraphs: c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\causal_graph\sub_causal_graph | exists=True
          importance: c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\importance_df | exists=True
                rows: c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\rows | exists=True
knowledge_graph_runs: c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\knowledge_graph | exists=True
        llm_response: c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\llm_response | exists=True
            articles: c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\articles | exists=False
  validation_

## 2. Helper functions reused/adapted from the existing notebooks

These functions collect the core logic used to parse event triplets, extract attribute categories, filter the causal graph by event-graph features, and reconstruct the unified causal-event graph.

In [3]:
def drop_unnamed_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed')].copy()


def read_csv_safe(path: Path) -> pd.DataFrame:
    return drop_unnamed_columns(pd.read_csv(path))


def parse_row_number(path: Path) -> Optional[int]:
    nums = re.findall(r'(\d+)', Path(path).stem)
    return int(nums[-1]) if nums else None


def parse_refined_triplet(value: Any) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None, None, None
    if isinstance(value, (list, tuple)) and len(value) >= 3:
        return str(value[0]), str(value[1]), str(value[2])
    s = str(value).strip()
    if not s or s.lower() == 'nan':
        return None, None, None
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)) and len(parsed) >= 3:
            return str(parsed[0]), str(parsed[1]), str(parsed[2])
    except Exception:
        pass
    for sep in ['->', '→']:
        if sep in s:
            parts = [p.strip() for p in s.split(sep)]
            if len(parts) >= 3:
                return parts[0], parts[1], parts[2]
    return None, None, None


def extract_attribute_categories(value: Any) -> List[str]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    s = str(value)
    cats = re.findall(r':\s*([^",\]\}]+)', s)
    cats = [c.strip() for c in cats if c and c.strip()]
    if not cats and s.strip() and s.strip().lower() != 'nan':
        cats = [s.strip()]
    return sorted(set(cats))


def filter_sub_causal_graph_by_features(clean_sub_causal_graph: pd.DataFrame, enriched_kg_df: pd.DataFrame) -> pd.DataFrame:
    feature_set = set(enriched_kg_df.get('feature', pd.Series(dtype=str)).dropna().astype(str))
    if not {'source', 'target'}.issubset(clean_sub_causal_graph.columns):
        raise ValueError('Causal graph must contain source and target columns.')
    mask = clean_sub_causal_graph['source'].astype(str).isin(feature_set) | clean_sub_causal_graph['target'].astype(str).isin(feature_set)
    return clean_sub_causal_graph.loc[mask].reset_index(drop=True)


def build_unified_graph_minimal(filtered_causal_df: pd.DataFrame, enriched_kg_df: pd.DataFrame,
                                triplet_col: str = 'refined_triplet', attribute_col: str = 'attribute') -> pd.DataFrame:
    unified_edges = []
    if filtered_causal_df.empty:
        return pd.DataFrame(columns=['source','relation','target','edge_type','weight','date','title','attribute'])

    for src in filtered_causal_df['source'].dropna().astype(str).unique():
        event_rows = enriched_kg_df.loc[enriched_kg_df.get('feature', pd.Series(dtype=str)).astype(str) == src]
        for _, row in event_rows.iterrows():
            ev_src, relation, ev_tgt = parse_refined_triplet(row.get(triplet_col))
            if ev_src is None:
                continue
            unified_edges.append({
                'source': ev_src, 'relation': relation, 'target': ev_tgt, 'edge_type': 'event',
                'weight': None, 'date': row.get('date'), 'title': row.get('title'), 'attribute': row.get(attribute_col),
            })
            for cat in extract_attribute_categories(row.get(attribute_col)):
                unified_edges.append({
                    'source': src, 'relation': 'has_attribute', 'target': cat, 'edge_type': 'attribute',
                    'weight': None, 'date': row.get('date'), 'title': row.get('title'), 'attribute': row.get(attribute_col),
                })
        for _, row in filtered_causal_df.loc[filtered_causal_df['source'].astype(str) == src].iterrows():
            unified_edges.append({
                'source': row['source'], 'relation': 'causal_effect', 'target': row['target'], 'edge_type': 'causal',
                'weight': row.get('weight'), 'date': None, 'title': None, 'attribute': None,
            })
    return pd.DataFrame(unified_edges)

## 3. Discover available artifacts

In [4]:
def recursive_files(base: Path, pattern: str) -> List[Path]:
    return sorted(base.rglob(pattern)) if base.exists() else []

artifact_paths = {
    'enriched_kg_csv': recursive_files(PATHS['enriched_kg'], 'enriched_kg*.csv'),
    'sub_causal_csv': recursive_files(PATHS['causal_subgraphs'], 'sub_causal_graph*.csv'),
    'importance_csv': recursive_files(PATHS['importance'], 'importance_df*.csv'),
    'row_csv': recursive_files(PATHS['rows'], 'row_*.csv'),
    'llm_response_json': recursive_files(PATHS['llm_response'], '*.json') + recursive_files(PATHS['knowledge_graph_runs'], '*.json'),
    'article_json': recursive_files(PATHS['articles'], '*.json'),
}

for k, files in artifact_paths.items():
    print(f'{k:>18}: {len(files)} files')
    for p in files[:5]:
        print('   ', p)
    if len(files) > 5:
        print('    ...')

   enriched_kg_csv: 3 files
    c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\enriched_knowledge_graph\enriched_kg_20260328_152610_936.csv
    c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\enriched_knowledge_graph\enriched_kg_20260331_112231_989.csv
    c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\enriched_knowledge_graph\enriched_kg_20260402_121459_548.csv
    sub_causal_csv: 4 files
    c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\causal_graph\sub_causal_graph\sub_causal_graph_1047.csv
    c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\causal_graph\sub_causal_graph\sub_causal_graph_548.csv
    c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\causal_graph\sub_causal_graph\sub_causal_graph_936.csv
    c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\causal_graph\sub_causal_graph\sub_c

## 4. Event graph coverage across saved runs

This summary is the first validation table: it reports all saved enriched event graphs and their feature/article/triplet coverage.

In [5]:
def summarize_enriched_kg(path: Path) -> Dict[str, Any]:
    df = read_csv_safe(path)
    feature_col = 'feature' if 'feature' in df.columns else None
    triplet_col = 'refined_triplet' if 'refined_triplet' in df.columns else None
    attribute_col = 'attribute' if 'attribute' in df.columns else None
    date_col = 'date' if 'date' in df.columns else None
    title_col = 'title' if 'title' in df.columns else None
    url_col = 'url' if 'url' in df.columns else None

    triplets = []
    if triplet_col:
        for v in df[triplet_col]:
            t = parse_refined_triplet(v)
            if t[0] is not None:
                triplets.append(t)
    categories = []
    if attribute_col:
        for v in df[attribute_col]:
            categories.extend(extract_attribute_categories(v))
    dates = pd.to_datetime(df[date_col], errors='coerce') if date_col else pd.Series(dtype='datetime64[ns]')

    return {
        'row_number': parse_row_number(path),
        'file': path.name,
        'rows_in_enriched_kg': len(df),
        'unique_features_with_event_evidence': df[feature_col].nunique() if feature_col else np.nan,
        'unique_event_triplets': len(set(triplets)),
        'unique_event_entities': len(set(t[0] for t in triplets)),
        'unique_event_relations': len(set(t[1] for t in triplets)),
        'unique_attribute_categories': len(set(categories)),
        'unique_articles_by_title': df[title_col].nunique() if title_col else np.nan,
        'unique_articles_by_url': df[url_col].nunique() if url_col else np.nan,
        'min_event_date': dates.min().date().isoformat() if len(dates) and pd.notna(dates.min()) else None,
        'max_event_date': dates.max().date().isoformat() if len(dates) and pd.notna(dates.max()) else None,
    }

kg_summary = pd.DataFrame([summarize_enriched_kg(p) for p in artifact_paths['enriched_kg_csv']])
kg_summary_path = PATHS['validation_outputs'] / 'event_graph_coverage_summary.csv'
kg_summary.to_csv(kg_summary_path, index=False)
print('Saved:', kg_summary_path)
kg_summary

Saved: c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\validation_outputs\event_graph_coverage_summary.csv


,row_number,file,rows_in_enriched_kg,unique_features_with_event_evidence,unique_event_triplets,unique_event_entities,unique_event_relations,unique_attribute_categories,unique_articles_by_title,unique_articles_by_url,min_event_date,max_event_date
0,936,enriched_kg_20260328_152610_936.csv,19,3,0,0,0,7,15,15,2023-01-12,2023-06-20
1,989,enriched_kg_20260331_112231_989.csv,15,2,0,0,0,4,13,13,2023-01-01,2023-06-18
2,548,enriched_kg_20260402_121459_548.csv,80,3,0,0,0,8,55,65,2023-01-01,2023-07-03


## 5. Causal-event graph validation across available cases

For each saved enriched event graph, this reconstructs the corresponding causal-event graph if the matching sub-causal graph exists.

In [6]:
def index_by_row(paths: Iterable[Path]) -> Dict[int, Path]:
    out = {}
    for p in paths:
        r = parse_row_number(p)
        if r is not None:
            out[r] = p
    return out

subgraph_by_row = index_by_row(artifact_paths['sub_causal_csv'])
rowfile_by_row = index_by_row(artifact_paths['row_csv'])
importance_by_row = index_by_row(artifact_paths['importance_csv'])


def infer_reference_date(row_number: Optional[int], enriched_df: pd.DataFrame) -> Optional[pd.Timestamp]:
    if row_number is not None and row_number in rowfile_by_row:
        row_df = read_csv_safe(rowfile_by_row[row_number])
        for col in ['Date', 'date', 'timestamp', 'Time', 'datetime']:
            if col in row_df.columns:
                d = pd.to_datetime(row_df[col].iloc[0], errors='coerce')
                if pd.notna(d):
                    return d
    if 'date' in enriched_df.columns:
        dates = pd.to_datetime(enriched_df['date'], errors='coerce')
        if dates.notna().any():
            return dates.max()
    return None


def summarize_unified_case(enriched_path: Path, windows=(7, 14, 21, 35, 60, 90)) -> Dict[str, Any]:
    kg_df = read_csv_safe(enriched_path)
    row_number = parse_row_number(enriched_path)
    causal_path = subgraph_by_row.get(row_number)
    result = {
        'row_number': row_number,
        'enriched_kg_file': enriched_path.name,
        'sub_causal_graph_file': causal_path.name if causal_path else None,
        'has_matching_causal_graph': causal_path is not None,
    }
    if causal_path is None:
        return result

    causal_df = read_csv_safe(causal_path)
    filtered_causal_df = filter_sub_causal_graph_by_features(causal_df, kg_df)
    unified_df = build_unified_graph_minimal(filtered_causal_df, kg_df)
    edge_counts = unified_df['edge_type'].value_counts().to_dict() if not unified_df.empty else {}
    nodes = set(unified_df.get('source', pd.Series(dtype=str)).dropna().astype(str)) | set(unified_df.get('target', pd.Series(dtype=str)).dropna().astype(str))
    features_with_event = set(kg_df.get('feature', pd.Series(dtype=str)).dropna().astype(str))
    causal_vars = set(causal_df.get('source', pd.Series(dtype=str)).dropna().astype(str)) | set(causal_df.get('target', pd.Series(dtype=str)).dropna().astype(str))
    retained_causal_vars = set(filtered_causal_df.get('source', pd.Series(dtype=str)).dropna().astype(str)) | set(filtered_causal_df.get('target', pd.Series(dtype=str)).dropna().astype(str))

    result.update({
        'event_graph_features': len(features_with_event),
        'full_sub_causal_edges': len(causal_df),
        'retained_causal_edges_after_feature_overlap': len(filtered_causal_df),
        'full_sub_causal_variables': len(causal_vars),
        'retained_causal_variables_after_feature_overlap': len(retained_causal_vars),
        'unified_nodes': len(nodes),
        'unified_event_edges': int(edge_counts.get('event', 0)),
        'unified_attribute_edges': int(edge_counts.get('attribute', 0)),
        'unified_causal_edges': int(edge_counts.get('causal', 0)),
    })
    ref_date = infer_reference_date(row_number, kg_df)
    result['reference_date'] = ref_date.date().isoformat() if ref_date is not None and pd.notna(ref_date) else None
    if ref_date is not None and pd.notna(ref_date) and not unified_df.empty and 'date' in unified_df.columns:
        event_edges = unified_df[unified_df['edge_type'].eq('event')].copy()
        event_edges['_date'] = pd.to_datetime(event_edges['date'], errors='coerce')
        for w in windows:
            start = ref_date - pd.Timedelta(days=w)
            mask = event_edges['_date'].notna() & (event_edges['_date'] <= ref_date) & (event_edges['_date'] >= start)
            result[f'event_edges_within_{w}d'] = int(mask.sum())
            result[f'event_nodes_within_{w}d'] = int(event_edges.loc[mask, 'source'].nunique())
    return result

case_summary = pd.DataFrame([summarize_unified_case(p) for p in artifact_paths['enriched_kg_csv']])
case_summary_path = PATHS['validation_outputs'] / 'causal_event_graph_case_summary.csv'
case_summary.to_csv(case_summary_path, index=False)
print('Saved:', case_summary_path)
case_summary

Saved: c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\validation_outputs\causal_event_graph_case_summary.csv


,row_number,enriched_kg_file,sub_causal_graph_file,has_matching_causal_graph,event_graph_features,full_sub_causal_edges,retained_causal_edges_after_feature_overlap,full_sub_causal_variables,retained_causal_variables_after_feature_overlap,unified_nodes,unified_event_edges,unified_attribute_edges,unified_causal_edges,reference_date,event_edges_within_7d,event_nodes_within_7d,event_edges_within_14d,event_nodes_within_14d,event_edges_within_21d,event_nodes_within_21d,event_edges_within_35d,event_nodes_within_35d,event_edges_within_60d,event_nodes_within_60d,event_edges_within_90d,event_nodes_within_90d
0,936,enriched_kg_20260328_152610_936.csv,sub_causal_graph_936.csv,True,3,24,2,16,3,3,0,0,2,2023-06-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,989,enriched_kg_20260331_112231_989.csv,sub_causal_graph_989.csv,True,2,22,0,15,0,0,0,0,0,2023-06-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,548,enriched_kg_20260402_121459_548.csv,sub_causal_graph_548.csv,True,3,22,3,16,5,5,0,0,3,2023-07-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 6. Model-important feature coverage

This table checks whether top local feature-importance outputs are covered by Event Registry-derived evidence.

In [7]:
def infer_feature_and_importance_columns(df: pd.DataFrame) -> Tuple[Optional[str], Optional[str]]:
    candidates_feature = ['feature', 'feature_name', 'Feature', 'name', 'variable', 'Variable']
    candidates_importance = ['importance', 'abs_shap', 'shap_value', 'SHAP', 'value', 'abs_value', 'mean_abs_shap']
    feature_col = next((c for c in candidates_feature if c in df.columns), None)
    importance_col = next((c for c in candidates_importance if c in df.columns), None)
    if feature_col is None:
        obj_cols = [c for c in df.columns if df[c].dtype == 'object']
        feature_col = obj_cols[0] if obj_cols else None
    if importance_col is None:
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        importance_col = num_cols[0] if num_cols else None
    return feature_col, importance_col


def top_features_from_importance(path: Path, top_n: int = 20) -> List[str]:
    df = read_csv_safe(path)
    feature_col, importance_col = infer_feature_and_importance_columns(df)
    if feature_col is None:
        return []
    work = df.copy()
    if importance_col is not None:
        work['_sort_importance'] = pd.to_numeric(work[importance_col], errors='coerce').abs()
        work = work.sort_values('_sort_importance', ascending=False)
    return [str(x) for x in work[feature_col].dropna().head(top_n).tolist()]


def feature_coverage_for_case(enriched_path: Path, top_n_values=(5, 10, 20)) -> Dict[str, Any]:
    kg_df = read_csv_safe(enriched_path)
    row_number = parse_row_number(enriched_path)
    event_features = set(kg_df.get('feature', pd.Series(dtype=str)).dropna().astype(str))
    imp_path = importance_by_row.get(row_number)
    out = {'row_number': row_number, 'enriched_kg_file': enriched_path.name, 'importance_file': imp_path.name if imp_path else None, 'event_graph_features': len(event_features)}
    if imp_path is None:
        return out
    for n in top_n_values:
        top_feats = top_features_from_importance(imp_path, top_n=n)
        overlap = set(top_feats) & event_features
        out[f'top_{n}_features'] = len(top_feats)
        out[f'top_{n}_covered_by_event_graph'] = len(overlap)
        out[f'top_{n}_coverage_ratio'] = round(len(overlap) / len(top_feats), 3) if top_feats else np.nan
        out[f'top_{n}_covered_features'] = ', '.join(sorted(overlap))
    return out

feature_coverage = pd.DataFrame([feature_coverage_for_case(p) for p in artifact_paths['enriched_kg_csv']])
feature_coverage_path = PATHS['validation_outputs'] / 'feature_importance_event_coverage.csv'
feature_coverage.to_csv(feature_coverage_path, index=False)
print('Saved:', feature_coverage_path)
feature_coverage

Saved: c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\validation_outputs\feature_importance_event_coverage.csv


,row_number,enriched_kg_file,importance_file,event_graph_features,top_5_features,top_5_covered_by_event_graph,top_5_coverage_ratio,top_5_covered_features,top_10_features,top_10_covered_by_event_graph,top_10_coverage_ratio,top_10_covered_features,top_20_features,top_20_covered_by_event_graph,top_20_coverage_ratio,top_20_covered_features
0,936,enriched_kg_20260328_152610_936.csv,importance_df_936.csv,3,5,0,0.0,,10,0,0.0,,20,3,0.15,"balance_of_trade, currency_log_return, stock_m..."
1,989,enriched_kg_20260331_112231_989.csv,importance_df_989.csv,2,5,0,0.0,,10,0,0.0,,20,2,0.10,"currency_log_return, stock_market_close"
2,548,enriched_kg_20260402_121459_548.csv,importance_df_548.csv,3,5,0,0.0,,10,0,0.0,,20,3,0.15,"balance_of_trade, imports, stock_market_close"


## 7. LLM response and LLM-as-Judge confirmed-link statistics

The current pipeline saves response JSON files only for articles with confirmed links. Therefore, these files can summarize confirmed links, but not full rejection rates. Use the optional audit in Section 9 for rejection-rate reporting.

In [8]:
def load_json_safe(path: Path) -> Optional[dict]:
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception:
        return None


def flatten_llm_response(path: Path) -> List[Dict[str, Any]]:
    data = load_json_safe(path)
    if not isinstance(data, dict):
        return []
    article = data.get('article', {}) or {}
    rows = []
    for conn in data.get('llm_connections', []) or []:
        ver = conn.get('verification', {}) or {}
        rows.append({
            'response_file': path.name,
            'response_folder': str(path.parent),
            'article_source': article.get('source'),
            'article_date': article.get('date'),
            'article_url': article.get('url'),
            'feature': conn.get('feature'),
            'event_summary': conn.get('event_summary'),
            'edge_label': conn.get('edge_label'),
            'evidence_quote': conn.get('evidence_quote'),
            'scope_justification': conn.get('scope_justification'),
            'quote_similarity': ver.get('quote_similarity'),
            'judge_verdict': ver.get('judge_verdict'),
            'judge_reasoning': ver.get('judge_reasoning'),
            'features_queried': ', '.join(data.get('features_queried', []) or []),
            'num_connections_found_in_file': data.get('num_connections_found'),
        })
    return rows

llm_rows = []
for p in artifact_paths['llm_response_json']:
    llm_rows.extend(flatten_llm_response(p))
llm_links = pd.DataFrame(llm_rows)

if llm_links.empty:
    llm_confirmed_summary = pd.DataFrame([{
        'response_files_with_confirmed_links': 0,
        'confirmed_event_feature_links': 0,
        'unique_features_confirmed': 0,
        'unique_articles_confirmed': 0,
        'mean_quote_similarity': np.nan,
        'min_quote_similarity': np.nan,
    }])
else:
    q = pd.to_numeric(llm_links.get('quote_similarity'), errors='coerce')
    llm_confirmed_summary = pd.DataFrame([{
        'response_files_with_confirmed_links': llm_links['response_file'].nunique(),
        'confirmed_event_feature_links': len(llm_links),
        'unique_features_confirmed': llm_links['feature'].nunique(),
        'unique_articles_confirmed': llm_links[['article_source','article_date','article_url']].drop_duplicates().shape[0],
        'mean_quote_similarity': round(q.mean(), 3) if q.notna().any() else np.nan,
        'min_quote_similarity': round(q.min(), 3) if q.notna().any() else np.nan,
    }])

llm_links_path = PATHS['validation_outputs'] / 'llm_confirmed_links.csv'
llm_summary_path = PATHS['validation_outputs'] / 'llm_confirmed_summary.csv'
llm_links.to_csv(llm_links_path, index=False)
llm_confirmed_summary.to_csv(llm_summary_path, index=False)
print('Saved:', llm_links_path)
print('Saved:', llm_summary_path)
llm_confirmed_summary

Saved: c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\validation_outputs\llm_confirmed_links.csv
Saved: c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\validation_outputs\llm_confirmed_summary.csv


,response_files_with_confirmed_links,confirmed_event_feature_links,unique_features_confirmed,unique_articles_confirmed,mean_quote_similarity,min_quote_similarity
0,104,114,4,79,0.99,0.768


## 8. Export paper-ready LaTeX tables

In [9]:
def compact_case_table(case_summary: pd.DataFrame) -> pd.DataFrame:
    keep = [
        'row_number', 'event_graph_features', 'full_sub_causal_edges',
        'retained_causal_edges_after_feature_overlap', 'unified_event_edges',
        'unified_attribute_edges', 'unified_causal_edges', 'unified_nodes', 'reference_date',
        'event_edges_within_35d'
    ]
    keep = [c for c in keep if c in case_summary.columns]
    return case_summary[keep].copy() if not case_summary.empty else pd.DataFrame()

paper_case_table = compact_case_table(case_summary)
case_tex = PATHS['validation_outputs'] / 'table_causal_event_validation.tex'
coverage_tex = PATHS['validation_outputs'] / 'table_feature_event_coverage.tex'
llm_tex = PATHS['validation_outputs'] / 'table_llm_judge_confirmed_summary.tex'

if not paper_case_table.empty:
    paper_case_table.to_latex(case_tex, index=False, escape=False,
                              caption='Validation summary of causal-event graph construction across saved prediction instances.',
                              label='tab:causal_event_validation')
if not feature_coverage.empty:
    feature_coverage.to_latex(coverage_tex, index=False, escape=False,
                              caption='Coverage of model-important features by Event Registry-based event evidence.',
                              label='tab:feature_event_coverage')
if not llm_confirmed_summary.empty:
    llm_confirmed_summary.to_latex(llm_tex, index=False, escape=False,
                                   caption='Summary of confirmed event-feature links from saved LLM-as-Judge response files.',
                                   label='tab:llm_judge_confirmed_summary')

print('LaTeX outputs:')
for p in [case_tex, coverage_tex, llm_tex]:
    print(' ', p, '| exists=', p.exists())
paper_case_table

LaTeX outputs:
  c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\validation_outputs\table_causal_event_validation.tex | exists= True
  c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\validation_outputs\table_feature_event_coverage.tex | exists= True
  c:\Users\Iman\OneDrive\Desktop\knowledge_graph_project\Experiment_materials\validation_outputs\table_llm_judge_confirmed_summary.tex | exists= True


,row_number,event_graph_features,full_sub_causal_edges,retained_causal_edges_after_feature_overlap,unified_event_edges,unified_attribute_edges,unified_causal_edges,unified_nodes,reference_date,event_edges_within_35d
0,936,3,24,2,0,0,2,3,2023-06-20,0.0
1,989,2,22,0,0,0,0,0,2023-06-18,NaN
2,548,3,22,3,0,0,3,5,2023-07-03,0.0


## 9. Optional audit-enabled LLM-as-Judge re-run via OpenRouter

This section re-runs the article extraction and judge verification pipeline on a fixed article sample using OpenRouter. It is disabled by default to avoid unexpected API costs. To run it, set `RUN_LLM_AUDIT=True` and provide `openrouter_api_key` or set the `OPENROUTER_API_KEY` environment variable.

The audit writes two files:

- `llm_as_judge_candidate_audit_log.csv`: one row per extracted candidate relation, with quote-check and judge outcomes.
- `llm_as_judge_article_audit_log.csv`: one row per article, including candidate and confirmed-link counts.

These outputs can be used to report rejection rates and article-level coverage beyond one illustrative case.


In [11]:
# Optional LLM-as-Judge audit using OpenRouter
# -----------------------------------------------------------------------------
# Why this is disabled by default:
# - Running this cell calls the LLM API repeatedly.
# - Set RUN_LLM_AUDIT=True only when you intentionally want to spend API credits.
# - For OpenRouter, provide your key either as OPENROUTER_API_KEY or as openrouter_api_key.

RUN_LLM_AUDIT = True  # Change to True to call OpenRouter.
AUDIT_ARTICLE_LIMIT = 50
AUDIT_RANDOM_SAMPLE = False
AUDIT_RANDOM_SEED = 42
AUDIT_MODEL = "openai/gpt-5-mini"  # Example: "openai/gpt-5-mini" or another OpenRouter model.
AUDIT_SLEEP_SECONDS = 0.0  # Increase if rate limited.
AUDIT_OUTPUT_DIR = PATHS['validation_outputs'] / 'llm_audit_run_openrouter'
AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")


# Option 1: set this variable directly in the notebook.
# openrouter_api_key = "sk-or-..."

# Option 2: set environment variable before launching Jupyter:
# export OPENROUTER_API_KEY="sk-or-..."


def get_openrouter_api_key() -> str:
    """Read OpenRouter key from notebook variable or environment."""
    key = globals().get('openrouter_api_key') or os.getenv('OPENROUTER_API_KEY')
    if not key:
        raise RuntimeError(
            "OpenRouter key not found. Define openrouter_api_key='sk-or-...' in a cell "
            "or set the OPENROUTER_API_KEY environment variable."
        )
    return key


def make_openrouter_client():
    """Create OpenAI-compatible client configured for OpenRouter."""
    from openai import OpenAI
    return OpenAI(
        api_key=get_openrouter_api_key(),
        base_url="https://openrouter.ai/api/v1",
    )


# Feature definitions used in the extraction prompt. These definitions matter because
# the extraction prompt enforces scope matching against the feature definition.
DEFAULT_FEATURE_DEFINITIONS = {
    "Product": "Product node identifier in the SupplyGraph dataset.",
    "Group": "Product group category.",
    "Sub-Group": "Product sub-group category.",
    "Production": "Daily manufacturing output quantity.",
    "Factory_Issue": "Daily quantity issued from the factory, representing manufacturing outflow or factory-related issue quantity.",
    "Delivery": "Daily quantity delivered to distributors, representing fulfilled shipments.",
    "Sales_lag_1": "Sales value one day earlier for the same product.",
    "Sales_lag_2": "Sales value two days earlier for the same product.",
    "Sales_lag_3": "Sales value three days earlier for the same product.",
    "Sales_lag_7": "Sales value seven days earlier, capturing weekly seasonality.",
    "Sales_lag_14": "Sales value fourteen days earlier, capturing bi-weekly effects.",
    "Sales_lag_21": "Sales value twenty-one days earlier, capturing three-week cycle effects.",
    "Sales_lag_28": "Sales value twenty-eight days earlier, capturing approximate monthly or four-week cycle effects.",
    "production_sales_ratio": "Production divided by sales plus epsilon; captures production volume relative to realized sales demand.",
    "delivery_sales_ratio": "Delivery divided by sales plus epsilon; captures downstream fulfillment relative to sales demand.",
    "factory_issue_rate": "Factory issue quantity divided by production plus epsilon; captures production-normalized issue intensity.",
    "inventory_proxy": "Cumulative production minus sales for each product; approximates inventory accumulation or depletion.",
    "interest_rate_value": "Benchmark interest rate level for Bangladesh.",
    "stock_market_close": "Closing value of the Bangladesh stock market index, representing market-level financial conditions.",
    "inflation_rate": "Monthly inflation rate, representing national consumer price change.",
    "exports": "Aggregate export value level for Bangladesh.",
    "imports": "Aggregate import value level for Bangladesh, representing the value of goods purchased from foreign entities.",
    "balance_of_trade": "National trade balance, reflecting the difference between exports and imports.",
    "currency_log_return": "Daily log return of USD/BDT exchange rate, representing day-to-day foreign-exchange movement.",
    "currency_momentum_5d": "Five-day momentum or trend in the USD/BDT exchange rate.",
    "holiday_type": "Holiday category, such as national holiday or non-holiday.",
    "holiday_name": "Name of the holiday or occasion.",
    "Rainfall": "Daily rainfall for Dhaka, Bangladesh.",
    "Sunshine": "Daily sunshine duration for Dhaka, Bangladesh.",
    "Humidity": "Daily humidity for Dhaka, Bangladesh.",
    "Temperature": "Daily temperature for Dhaka, Bangladesh.",
    "category": "Structured event category on a given date, such as cultural, religious, or no event.",
    "scope_or_location": "Geographic scope or location of the structured event, such as nationwide or urban centers.",
    "typical_rain_mm": "Typical rainfall in millimeters associated with the event context.",
}


def choose_audit_features(max_features: int = 10) -> List[str]:
    """
    Choose features for the audit.
    Priority:
    1. User-defined AUDIT_FEATURES list, if present.
    2. Top features from the first importance CSV, if available.
    3. Features appearing in the first enriched event graph, if available.
    4. A conservative fallback list used in the current study's example.
    """
    if 'AUDIT_FEATURES' in globals() and globals()['AUDIT_FEATURES']:
        return list(globals()['AUDIT_FEATURES'])[:max_features]

    if artifact_paths.get('importance_csv'):
        try:
            feats = top_features_from_importance(artifact_paths['importance_csv'][0], top_n=max_features)
            if feats:
                return feats[:max_features]
        except Exception as e:
            print(f"Could not load top features from importance file: {e}")

    if artifact_paths.get('enriched_kg_csv'):
        try:
            kg = read_csv_safe(artifact_paths['enriched_kg_csv'][0])
            if 'feature' in kg.columns:
                feats = sorted(kg['feature'].dropna().astype(str).unique().tolist())
                if feats:
                    return feats[:max_features]
        except Exception as e:
            print(f"Could not load features from enriched KG: {e}")

    return ['imports', 'balance_of_trade', 'stock_market_close'][:max_features]


def build_audit_feature_map(max_features: int = 10) -> Dict[str, str]:
    """Build feature map with definitions for the extraction prompt."""
    features = choose_audit_features(max_features=max_features)
    return {
        f: DEFAULT_FEATURE_DEFINITIONS.get(f, f"Supply-chain or contextual feature used in the forecasting and explanation pipeline: {f}")
        for f in features
    }


def verify_quote_in_article(evidence_quote: str, article_body: str, threshold: float = 0.60) -> Tuple[bool, str, float]:
    """
    Layer 1: programmatic quote check.
    Returns (is_valid, best_match_snippet, similarity_score).
    """
    if not evidence_quote or not article_body:
        return False, '', 0.0

    quote_lower = str(evidence_quote).lower().strip()
    body_words_original = str(article_body).split()
    body_words_lower = [w.lower() for w in body_words_original]
    quote_words = quote_lower.split()
    if not quote_words:
        return False, '', 0.0

    quote_len = len(quote_words)
    window_min = max(1, int(quote_len * 0.70))
    window_max = min(len(body_words_lower), int(quote_len * 1.30) + 1)

    best_score, best_snippet = 0.0, ''
    for win_size in range(window_min, window_max + 1):
        for start in range(0, len(body_words_lower) - win_size + 1):
            candidate_lower = ' '.join(body_words_lower[start:start + win_size])
            score = difflib.SequenceMatcher(None, quote_lower, candidate_lower).ratio()
            if score > best_score:
                best_score = score
                best_snippet = ' '.join(body_words_original[start:start + win_size])

    return best_score >= threshold, best_snippet, round(best_score, 3)


def load_articles_from_folder(articles_folder: Path) -> List[dict]:
    """Load Event Registry article JSON files from PATHS['articles']."""
    articles = []
    for p in recursive_files(articles_folder, '*.json'):
        data = load_json_safe(p)
        if not isinstance(data, dict):
            continue

        # Common format: {'articles': [...]}.
        if isinstance(data.get('articles'), list):
            source_articles = data.get('articles', [])
        # Alternative: a single article object.
        elif any(k in data for k in ['body', 'title', 'date', 'url']):
            source_articles = [data]
        else:
            source_articles = []

        for a in source_articles:
            if not isinstance(a, dict):
                continue
            articles.append({
                'title': a.get('title'),
                'url': a.get('url'),
                'source': a.get('source') or a.get('sourceTitle') or a.get('provider'),
                'body': a.get('body') or a.get('text') or a.get('content'),
                'date': a.get('date') or a.get('publishedAt') or a.get('dateTime'),
                'file': str(p),
            })
    return articles


def select_audit_articles(articles: List[dict], limit: int) -> List[dict]:
    """Select non-empty article bodies for audit."""
    usable = [a for a in articles if a.get('body') and len(str(a.get('body'))) >= 100]
    if AUDIT_RANDOM_SAMPLE:
        rng = np.random.default_rng(AUDIT_RANDOM_SEED)
        idx = rng.choice(len(usable), size=min(limit, len(usable)), replace=False)
        return [usable[i] for i in idx]
    return usable[:limit]


def call_json_llm(client, prompt: str, model: str = AUDIT_MODEL) -> dict:
    """Call OpenRouter with JSON-object response. Includes a fallback without response_format."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
        )
        return json.loads(response.choices[0].message.content)
    except Exception as first_error:
        # Some models/providers may not accept response_format. Try again without it.
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
            )
            content = response.choices[0].message.content
            # Try to recover JSON even if wrapped in text/code fences.
            match = re.search(r'\{.*\}', content, flags=re.DOTALL)
            if not match:
                raise ValueError(f"No JSON object found in response: {content[:300]}")
            return json.loads(match.group(0))
        except Exception as second_error:
            raise RuntimeError(f"LLM JSON call failed. First error: {first_error}; fallback error: {second_error}")


def extract_causal_triplets_with_llm(article_body: str, article_source: str, feature_map: Dict[str, str], client) -> List[dict]:
    """
    Strict extraction prompt adapted from the original 01_whole_pipeline.ipynb code.
    It asks the LLM to return candidate event-feature causal links only when grounded in the article.
    """
    prompt = f"""
You are a Supply Chain Causal Analyst for the Bangladesh Manufacturing Sector.
Your job is STRICT EXTRACTION — you may ONLY report causal relationships that are EXPLICITLY STATED in the article text below. You must NOT infer, assume, or use outside knowledge.

INPUT DATA:
1. SOURCE: {article_source}
2. NEWS TEXT: "{article_body}"
3. TARGET FEATURES (each has a NAME and a DEFINITION — both matter):
{json.dumps(feature_map, indent=2)}

TASK:
Determine if this specific news article EXPLICITLY describes a real-world event that causally impacts any of the Target Features.

STRICT RULES (follow every single one):

1. GROUNDING REQUIREMENT: Every connection you return MUST be backed by a direct quote or close paraphrase from the article. If the article does not explicitly mention the causal link, do NOT include it.

2. NO INFERENCE: Do NOT use your general knowledge to connect events to features. For example, if the article mentions "USD strengthened" but does NOT mention imports, trade balance, or supply chain costs, you must NOT link it to 'imports' or 'balance_of_trade' — even if such a link is plausible in theory.

3. NO HALLUCINATION: Every "event_summary" you return must correspond to a specific event described in the article. Do not fabricate or generalize events.

4. FEATURE NAME EXACTNESS: The "feature" field must be an EXACT key from the Target Features dictionary. Do not rename or paraphrase feature names.

5. RELEVANCE CHECK: If the article is unrelated to Bangladesh's economy, manufacturing, trade, or supply chain, return an empty list immediately.

6. CAUSALITY IN TEXT: The article must explicitly state or strongly imply a cause-and-effect relationship (e.g., "X led to Y", "X caused Y", "due to X, Y happened"). Mere co-mention of two topics is NOT sufficient.

7. SCOPE MATCHING (CRITICAL): Each Target Feature has a specific DEFINITION describing its scope and meaning. The causal relationship in the article must match THE SAME SCOPE as the feature definition. Apply this two-step check:
   Step A — Read the feature DEFINITION carefully (not just the feature name).
   Step B — Ask: "Does the article discuss this concept AT THE SAME SCOPE described in the definition?"
   REJECT the connection if there is a scope mismatch. Common scope mismatches to watch for:
     - Article discusses a SINGLE commodity (e.g., "LNG imports limited") but the feature definition refers to AGGREGATE/COUNTRY-WIDE metric (e.g., "total volume of goods brought in") → REJECT.
     - Article discusses one company's stock but the feature refers to the overall stock market index → REJECT.
     - Article discusses a regional event but the feature refers to a national-level indicator → REJECT.
   Only ACCEPT if the article explicitly discusses the concept at the same level of aggregation as the feature definition.

8. EDGE LABEL: Create a short, active verb phrase for the edge (e.g., "disrupts flow", "halts operations", "increases cost").

9. SUPPORTING EVIDENCE: For each connection, include a verbatim quote (max 30 words) from the article that proves the relationship exists.

10. SCOPE JUSTIFICATION: For each connection, briefly explain (1 sentence) why the article's claim matches the feature definition's scope — not just the feature name.

OUTPUT:
Return a strict JSON object with a list of "connections". If no valid, grounded connections exist, return {{"connections": []}}.
Format:
{{
    "connections": [
        {{
            "feature": "Exact Key from Target Features",
            "event_summary": "3-5 word summary of the news event",
            "edge_label": "active verb phrase description",
            "evidence_quote": "verbatim quote from article proving this link (max 30 words)",
            "scope_justification": "1 sentence explaining why the article's claim matches the feature definition's scope"
        }}
    ]
}}

FINAL CHECK before responding:
- For each connection, re-read the article and verify the evidence_quote actually appears in the text.
- For each connection, re-read the feature DEFINITION and confirm the article discusses that concept at the same scope (not a narrower subset or a single example).
- If the evidence_quote does not exist in the text, REMOVE that connection.
- If there is a scope mismatch, REMOVE that connection.
- It is better to return an empty list than to return a fabricated or scope-mismatched connection.
"""
    try:
        data = call_json_llm(client, prompt, AUDIT_MODEL)
        conns = data.get('connections', [])
        return conns if isinstance(conns, list) else []
    except Exception as e:
        print(f"LLM extraction error: {e}")
        return []


def judge_connection_with_llm(article_body: str, connection: dict, client) -> dict:
    """LLM-as-Judge verification prompt adapted from the original pipeline."""
    judge_prompt = f"""You are an impartial fact-checking judge. Your ONLY job is to verify whether a claimed causal relationship is EXPLICITLY supported by the article text below.

ARTICLE TEXT:
"{article_body}"

CLAIMED RELATIONSHIP:
Feature: {connection.get('feature', '')}
Event: {connection.get('event_summary', '')}
Causal link: {connection.get('edge_label', '')}
Evidence quote: "{connection.get('evidence_quote', '')}"
Scope justification: {connection.get('scope_justification', '')}

VERIFICATION TASKS

1. Quote check: Does the evidence quote, or a very close paraphrase, actually appear in the article?

2. Causality check: Does the article explicitly state or strongly imply a cause-and-effect relationship between the event and the feature? Mere co-mention is not enough.

3. Scope check: Does the article discuss this at the same scope or level as the feature implies, for example country-level versus single-company?

RULES

Base your judgment only on the article text provided. If any of the three checks fail, the verdict is rejected. Be strict; it is better to reject a borderline case than to confirm an unsupported one.

OUTPUT

Return a strict JSON object with the following fields:
{{ "verdict": "CONFIRMED" or "REJECTED", "quote_found": true/false, "causality_explicit": true/false, "scope_matches": true/false, "reasoning": "1-2 sentence explanation" }}
"""
    try:
        data = call_json_llm(client, judge_prompt, AUDIT_MODEL)
        return {
            'verdict': str(data.get('verdict', 'REJECTED')).upper(),
            'quote_found': bool(data.get('quote_found', False)),
            'causality_explicit': bool(data.get('causality_explicit', False)),
            'scope_matches': bool(data.get('scope_matches', False)),
            'reasoning': data.get('reasoning', ''),
        }
    except Exception as e:
        return {
            'verdict': 'REJECTED',
            'quote_found': False,
            'causality_explicit': False,
            'scope_matches': False,
            'reasoning': f'Judge error: {e}',
        }


def audit_single_article(article: dict, feature_map: Dict[str, str], client) -> Tuple[dict, List[dict]]:
    """Run extraction + quote check + LLM judge for one article."""
    body = str(article.get('body') or '')
    source = article.get('source') or 'Unknown Source'
    raw_connections = extract_causal_triplets_with_llm(body, source, feature_map, client)

    candidate_rows = []
    confirmed_count = 0
    rejected_quote_count = 0
    rejected_judge_count = 0

    for conn_idx, conn in enumerate(raw_connections):
        quote_ok, best_match, sim = verify_quote_in_article(conn.get('evidence_quote', ''), body)
        row = {
            'article_title': article.get('title'),
            'article_source': source,
            'article_date': article.get('date'),
            'article_url': article.get('url'),
            'article_file': article.get('file'),
            'candidate_index': conn_idx,
            'feature': conn.get('feature'),
            'event_summary': conn.get('event_summary'),
            'edge_label': conn.get('edge_label'),
            'evidence_quote': conn.get('evidence_quote'),
            'scope_justification': conn.get('scope_justification'),
            'quote_check_passed': quote_ok,
            'quote_similarity': sim,
            'quote_best_match': best_match,
            'judge_verdict': None,
            'quote_found': None,
            'causality_explicit': None,
            'scope_matches': None,
            'judge_reasoning': None,
            'final_verdict': None,
        }

        if not quote_ok:
            row['final_verdict'] = 'REJECTED_QUOTE_CHECK'
            rejected_quote_count += 1
            candidate_rows.append(row)
            continue

        judge = judge_connection_with_llm(body, conn, client)
        row.update({
            'judge_verdict': judge.get('verdict'),
            'quote_found': judge.get('quote_found'),
            'causality_explicit': judge.get('causality_explicit'),
            'scope_matches': judge.get('scope_matches'),
            'judge_reasoning': judge.get('reasoning'),
        })

        if judge.get('verdict') == 'CONFIRMED':
            row['final_verdict'] = 'CONFIRMED'
            confirmed_count += 1
        else:
            row['final_verdict'] = 'REJECTED_LLM_JUDGE'
            rejected_judge_count += 1
        candidate_rows.append(row)

        if AUDIT_SLEEP_SECONDS:
            import time
            time.sleep(AUDIT_SLEEP_SECONDS)

    article_row = {
        'article_title': article.get('title'),
        'article_source': source,
        'article_date': article.get('date'),
        'article_url': article.get('url'),
        'article_file': article.get('file'),
        'raw_candidate_links': len(raw_connections),
        'confirmed_links': confirmed_count,
        'rejected_quote_check': rejected_quote_count,
        'rejected_llm_judge': rejected_judge_count,
    }
    return article_row, candidate_rows


if RUN_LLM_AUDIT:
    client = make_openrouter_client()
    feature_map = build_audit_feature_map(max_features=10)
    print('Audit model:', AUDIT_MODEL)
    print('Audit features:')
    print(json.dumps(feature_map, indent=2))

    articles_all = load_articles_from_folder(PATHS['articles'])
    articles = select_audit_articles(articles_all, AUDIT_ARTICLE_LIMIT)
    if not articles:
        raise RuntimeError('No usable article bodies found. Check PATHS["articles"] and article JSON format.')

    all_article_rows = []
    all_candidate_rows = []

    for i, article in enumerate(articles, start=1):
        print(f'[{i}/{len(articles)}] {article.get("source")} | {article.get("date")} | {article.get("title")}')
        article_row, candidate_rows = audit_single_article(article, feature_map, client)
        all_article_rows.append(article_row)
        all_candidate_rows.extend(candidate_rows)
        if AUDIT_SLEEP_SECONDS:
            import time
            time.sleep(AUDIT_SLEEP_SECONDS)

    article_audit_df = pd.DataFrame(all_article_rows)
    candidate_audit_df = pd.DataFrame(all_candidate_rows)

    article_csv = AUDIT_OUTPUT_DIR / 'llm_as_judge_article_audit_log.csv'
    candidate_csv = AUDIT_OUTPUT_DIR / 'llm_as_judge_candidate_audit_log.csv'
    summary_csv = AUDIT_OUTPUT_DIR / 'llm_as_judge_audit_summary.csv'
    latex_csv = AUDIT_OUTPUT_DIR / 'table_llm_as_judge_audit_summary.tex'

    article_audit_df.to_csv(article_csv, index=False)
    candidate_audit_df.to_csv(candidate_csv, index=False)

    # Candidate-level rejection/confirmation summary.
    if not candidate_audit_df.empty:
        audit_summary = candidate_audit_df['final_verdict'].value_counts().rename_axis('outcome').reset_index(name='count')
        audit_summary['ratio'] = audit_summary['count'] / audit_summary['count'].sum()
    else:
        audit_summary = pd.DataFrame(columns=['outcome', 'count', 'ratio'])

    # Article-level coverage summary.
    article_summary = pd.DataFrame({
        'metric': [
            'articles_processed',
            'articles_with_candidate_links',
            'articles_with_confirmed_links',
            'raw_candidate_links',
            'confirmed_links',
            'rejected_quote_check',
            'rejected_llm_judge',
        ],
        'value': [
            len(article_audit_df),
            int((article_audit_df['raw_candidate_links'] > 0).sum()) if not article_audit_df.empty else 0,
            int((article_audit_df['confirmed_links'] > 0).sum()) if not article_audit_df.empty else 0,
            int(article_audit_df['raw_candidate_links'].sum()) if not article_audit_df.empty else 0,
            int(article_audit_df['confirmed_links'].sum()) if not article_audit_df.empty else 0,
            int(article_audit_df['rejected_quote_check'].sum()) if not article_audit_df.empty else 0,
            int(article_audit_df['rejected_llm_judge'].sum()) if not article_audit_df.empty else 0,
        ],
    })

    with open(summary_csv, 'w', encoding='utf-8') as f:
        f.write('# Candidate-level summary\n')
        audit_summary.to_csv(f, index=False)
        f.write('\n# Article-level summary\n')
        article_summary.to_csv(f, index=False)

    if not article_summary.empty:
        article_summary.to_latex(
            latex_csv,
            index=False,
            escape=False,
            caption='LLM-as-Judge audit summary across the sampled Event Registry articles.',
            label='tab:llm_as_judge_audit_summary',
        )

    print(f'Wrote article audit log to: {article_csv}')
    print(f'Wrote candidate audit log to: {candidate_csv}')
    print(f'Wrote audit summary to: {summary_csv}')
    print(f'Wrote LaTeX summary to: {latex_csv}')
    display(article_summary)
    display(audit_summary)
else:
    print('RUN_LLM_AUDIT=False. Skipping OpenRouter API calls.')
    print('To run the audit: set RUN_LLM_AUDIT=True and define openrouter_api_key or OPENROUTER_API_KEY.')


Audit model: openai/gpt-5-mini
Audit features:
{
  "Delivery": "Daily quantity delivered to distributors, representing fulfilled shipments.",
  "delivery_sales_ratio": "Delivery divided by sales plus epsilon; captures downstream fulfillment relative to sales demand.",
  "production_sales_ratio": "Production divided by sales plus epsilon; captures production volume relative to realized sales demand.",
  "Production": "Daily manufacturing output quantity.",
  "inventory_proxy": "Cumulative production minus sales for each product; approximates inventory accumulation or depletion.",
  "Sales_lag_28": "Sales value twenty-eight days earlier, capturing approximate monthly or four-week cycle effects.",
  "Factory_Issue": "Daily quantity issued from the factory, representing manufacturing outflow or factory-related issue quantity.",
  "Sales_lag_7": "Sales value seven days earlier, capturing weekly seasonality.",
  "Sales_lag_3": "Sales value three days earlier for the same product.",
  "Sales_

RuntimeError: No usable article bodies found. Check PATHS["articles"] and article JSON format.

## 10. Draft manuscript text

Use this draft paragraph after checking the generated tables.

In [ ]:
def draft_validation_text(case_summary: pd.DataFrame, kg_summary: pd.DataFrame, llm_summary: pd.DataFrame) -> str:
    n_cases = int(case_summary['has_matching_causal_graph'].sum()) if 'has_matching_causal_graph' in case_summary.columns else len(case_summary)
    n_kg = len(kg_summary)
    total_event_edges = int(case_summary.get('unified_event_edges', pd.Series(dtype=float)).fillna(0).sum()) if not case_summary.empty else 0
    total_attr_edges = int(case_summary.get('unified_attribute_edges', pd.Series(dtype=float)).fillna(0).sum()) if not case_summary.empty else 0
    total_causal_edges = int(case_summary.get('unified_causal_edges', pd.Series(dtype=float)).fillna(0).sum()) if not case_summary.empty else 0
    confirmed = int(llm_summary.get('confirmed_event_feature_links', pd.Series([0])).iloc[0]) if not llm_summary.empty else 0
    files = int(llm_summary.get('response_files_with_confirmed_links', pd.Series([0])).iloc[0]) if not llm_summary.empty else 0
    return f"""
To assess whether the proposed framework generalizes beyond a single illustrative graph, we summarized all saved explanation artifacts produced by the pipeline. Across {n_kg} enriched event-graph files, {n_cases} cases had matching sub-causal graphs and could be reconstructed as causal-event graphs. The reconstructed unified graphs contained {total_event_edges} event edges, {total_attr_edges} attribute edges, and {total_causal_edges} retained causal edges. In addition, the saved LLM response files contained {confirmed} confirmed event-feature links across {files} article-level response files. These summaries provide a reproducible validation layer for the framework by reporting how often model-important features can be linked to external event evidence, how much of the causal graph remains connected to the event layer, and how many event-feature relations survive the grounding and LLM-as-Judge verification process.
""".strip()

validation_text = draft_validation_text(case_summary, kg_summary, llm_confirmed_summary)
print(validation_text)
text_path = PATHS['validation_outputs'] / 'draft_validation_paragraph.txt'
text_path.write_text(validation_text, encoding='utf-8')
print('
Saved:', text_path)

## 11. Recommended paper insertion point

Insert the resulting validation table(s) after the current LLM-as-Judge subsection or immediately before the conclusion as:

```latex
\subsection{Framework Validation Across Saved Prediction Instances}
```

At minimum, include:

- `Table~
ef{tab:causal_event_validation}` for per-case causal-event graph statistics.
- `Table~
ef{tab:feature_event_coverage}` for coverage of model-important features.
- `Table~
ef{tab:llm_judge_confirmed_summary}` or the optional audit table if you run Section 9.

For a stronger Q1 submission, run the optional audit on a fixed article sample and report confirmed, quote-rejected, and judge-rejected counts.